In [ ]:
import os
import time
import gc
from datetime import datetime
from IPython import get_ipython
import pandas as pd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import threading
from openai import OpenAI

# Set your Perplexity API key here or via environment variable before running
os.environ["PERPLEXITY_API_KEY"] = "PERPLEXITY_API_KEY"

# Initialize Perplexity client with API base URL override
client = OpenAI(
    api_key=os.getenv("PERPLEXITY_API_KEY"),
    base_url="https://api.perplexity.ai"
)

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
EMAIL_SENDER = "email_sender@gmail.com"
EMAIL_PASSWORD = "your_email_password"
EMAIL_RECEIVER = "alert_receiver@gmail.com"
ALERT_THRESHOLD = 1

def send_email_in_background(subject, body):
    def _send_email():
        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        msg['To'] = EMAIL_RECEIVER
        msg['Subject'] = subject
        msg.attach(MIMEText(body, "plain"))
        try:
            server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.sendmail(EMAIL_SENDER, EMAIL_RECEIVER, msg.as_string())
            server.quit()
            print(f"Email sent: {subject}")
        except Exception as e:
            print(f"Failed to send email: {e}")

    threading.Thread(target=_send_email, daemon=True).start()

class NotebookCellAnalyzer:
    def __init__(self):
        self.current_cell = {}
        self.cell_counter = 0
        self.all_metrics = []
        self.slow_cells = []
        self.session_start_time = datetime.now()
        self._register_hooks()

    def _register_hooks(self):
        ip = get_ipython()
        if ip:
            ip.events.register('pre_run_cell', self._pre_run_cell)
            ip.events.register('post_run_cell', self._post_run_cell)

    def _pre_run_cell(self, info):
        self.cell_counter += 1
        gc.collect()
        pre_metrics = self._get_metrics()
        cell_content = str(info.raw_cell) if hasattr(info, 'raw_cell') else ""
        first_sentence = self._get_first_sentence(cell_content)
        self.current_cell = {
            'cell_number': self.cell_counter,
            'start_time': datetime.now(),
            'start_metrics': pre_metrics,
            'description': first_sentence
        }

    def _post_run_cell(self, result):
        if not self.current_cell:
            return

        end_time = datetime.now()
        execution_time = (end_time - self.current_cell['start_time']).total_seconds()

        if execution_time > ALERT_THRESHOLD:
            self.slow_cells.append({
                'cell_number': self.current_cell['cell_number'],
                'exec_time': execution_time,
                'description': self.current_cell['description']
            })

        gc.collect()
        post_metrics = self._get_metrics()
        memory_delta = post_metrics['memory_mb'] - self.current_cell['start_metrics']['memory_mb']

        metrics_record = {
            'cell_number': self.current_cell['cell_number'],
            'description': self.current_cell['description'],
            'start_time': self.current_cell['start_time'],
            'end_time': end_time,
            'execution_time_seconds': round(execution_time, 4),
            'start_memory_mb': round(self.current_cell['start_metrics']['memory_mb'], 2),
            'end_memory_mb': round(post_metrics['memory_mb'], 2),
            'memory_delta_mb': round(memory_delta, 2),
            'success': result.success if hasattr(result, 'success') else True,
            'timestamp': datetime.now()
        }

        self.all_metrics.append(metrics_record)
        self.current_cell = {}

    def _get_metrics(self):
        try:
            import psutil
            process = psutil.Process()
            memory_info = process.memory_info()
            return {'memory_mb': round(memory_info.rss / 1024 / 1024, 2)}
        except:
            return {'memory_mb': 0}

    def _get_first_sentence(self, cell_content):
        if not cell_content:
            return "Empty cell"
        lines = cell_content.strip().split('\n')
        for line in lines:
            line = line.strip()
            if not line:
                continue
            if line.startswith('#'):
                return line[1:].strip()[:100]
            if line.startswith('"""') or line.startswith("'''"):
                continue
            return line[:100]
        return "Empty cell"

    def get_dataframe(self):
        if not self.all_metrics:
            return None
        return pd.DataFrame(self.all_metrics).sort_values('cell_number')

    def show_summary(self):
        df = self.get_dataframe()
        if df is not None:
            print(f"Cells executed: {len(df)}")
            print(f"Total time: {df['execution_time_seconds'].sum():.2f}s")
            print(f"Average time per cell: {df['execution_time_seconds'].mean():.2f}s")
            print(f"Memory change: {df['memory_delta_mb'].sum():.1f}MB")
            print("\nTop 5 slowest cells:")
            print(df.nlargest(5, 'execution_time_seconds')[['cell_number', 'description', 'execution_time_seconds', 'memory_delta_mb']])

    def stop_tracking(self):
        ip = get_ipython()
        if ip:
            ip.events.unregister('pre_run_cell', self._pre_run_cell)
            ip.events.unregister('post_run_cell', self._post_run_cell)

    def clear_metrics(self):
        self.all_metrics = []
        self.slow_cells = []
        self.cell_counter = 0
        self.session_start_time = datetime.now()

    def send_all_alerts(self, llm_summary=None):
        if not self.slow_cells:
            print("No slow cells found.")
            return

        subject = f"Slow Cells Report - {len(self.slow_cells)} cells exceeded {ALERT_THRESHOLD}s"

        if llm_summary:
            body = llm_summary
        else:
            body_lines = []
            for cell in self.slow_cells:
                body_lines.append(f"Cell {cell['cell_number']} | {cell['exec_time']:.2f}s | {cell['description']}")
            body = "\n".join(body_lines)

        send_email_in_background(subject, body)
        print(f"Summary email sent with {len(self.slow_cells)} slow cells.")

# Enhanced summary function with solutions and approaches:
def llm_generate_summary(df):
    if df is None or df.empty:
        return "No cell metrics available to summarize."

    total_cells = len(df)
    total_time = round(df['execution_time_seconds'].sum(), 2)
    avg_time = round(df['execution_time_seconds'].mean(), 2)
    mem_delta = round(df['memory_delta_mb'].sum(), 2)
    slowest = df.nlargest(3, 'execution_time_seconds')[['cell_number', 'execution_time_seconds', 'memory_delta_mb', 'description']]
    slow_lines = []
    for _, row in slowest.iterrows():
        slow_lines.append(
            f"Cell {int(row['cell_number'])}: {row['execution_time_seconds']:.2f}s, Mem {row['memory_delta_mb']:.2f}MB — {row['description']}"
        )

    summary = []
    summary.append(" Notebook Performance Summary")
    summary.append("")
    summary.append("Key Metrics:")
    summary.append(f"  - Cells Executed: {total_cells}")
    summary.append(f"  - Total Time: {total_time}s")
    summary.append(f"  - Avg Time per Cell: {avg_time}s")
    summary.append(f"  - Net Memory Change: {mem_delta}MB")
    summary.append("")
    summary.append("Top 3 Slowest Cells:")
    summary.extend([f"  - {line}" for line in slow_lines])
    summary.append("")
    summary.append("Solutions & Approaches:")
    summary.append("  • For slowest cells:")
    summary.append("     - Solution: Profile cell with %timeit or cProfile and optimize slow code paths.")
    summary.append("     - Alternatives:")
    summary.append("         · Refactor to process data in batches instead of one-by-one.")
    summary.append("         · Consider using multiprocessing or Dask for CPU-bound operations.")
    summary.append("         · For pandas-heavy code, try vectorization or leveraging libraries like NumExpr, Polars, or PySpark if the workload is large.")
    summary.append("")
    summary.append("  • For high memory usage:")
    summary.append("     - Solution: Release unused variables early with 'del' and use gc.collect().")
    summary.append("     - Alternatives:")
    summary.append("         · Save intermediate large dataframes to disk (e.g., Parquet files), reload only when needed.")
    summary.append("         · Use pandas chunking (e.g., read_csv(..., chunksize=...)) for big datasets.")
    summary.append("         · Move heavy operations to cloud/distributed compute platforms if memory is a bottleneck.")
    summary.append("")
    summary.append("  • For overall efficiency:")
    summary.append("     - Solution: Group related operations and minimize cell state changes.")
    summary.append("     - Alternatives:")
    summary.append("         · Use Jupyter cell magics like %%capture to reduce logging overhead.")
    summary.append("         · Adopt modular code practices (functions/classes) to organize and scale workflow.")
    summary.append("")
    summary.append("(Generated by Perplexity AI bot.)")
    return "\n".join(summary)


# Initialize analyzer
cell_analyzer = NotebookCellAnalyzer()
#IMPORTING PACKAGES AND CREATING DATASET 1
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random


np.random.seed(42)
random.seed(42)



print("Creating E-commerce Dataset...")


n_customers = 100000
customers_data = {
    'customer_id': range(1, n_customers + 1),
    'first_name': [f'Customer_{i}' for i in range(1, n_customers + 1)],
    'last_name': [f'LastName_{i}' for i in range(1, n_customers + 1)],
    'email': [f'customer_{i}@email.com' for i in range(1, n_customers + 1)],
    'registration_date': pd.date_range('2020-01-01', periods=n_customers, freq='H'),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], n_customers),
    'customer_segment': np.random.choice(['Premium', 'Standard', 'Basic'], n_customers, p=[0.2, 0.5, 0.3])
}

customers_df = pd.DataFrame(customers_data)


n_orders = 5000000

probs = np.random.exponential(scale=0.5, size=n_customers)
probs = probs / probs.sum()
customer_order_distribution = np.random.choice(
    customers_df['customer_id'].values,
    size=n_orders,
    replace=True,
    p=probs
)

orders_data = {
    'order_id': range(1, n_orders + 1),
    'customer_id': customer_order_distribution,
    'order_date': pd.date_range('2020-01-01', periods=n_orders, freq='15T'),
    'total_amount': np.random.exponential(scale=100, size=n_orders).round(2),
    'order_status': np.random.choice(['Completed', 'Pending', 'Cancelled'], n_orders, p=[0.8, 0.15, 0.05]),
    'payment_method': np.random.choice(['Credit Card', 'PayPal', 'Bank Transfer'], n_orders),
    'shipping_cost': np.random.normal(10, 3, n_orders).round(2)
}

orders_df = pd.DataFrame(orders_data)

print(f"Customers DataFrame: {customers_df.shape}")
print(f"Orders DataFrame: {orders_df.shape}")
print(f"Join Key: customer_id")
print(f"Join Cardinality: 1-to-Many (One customer can have multiple orders)")
print()
#PERFORMING JOIN OPERATIONS ON DATASET 1
print("\n1. Inner Join Example (Customers with Orders):")
customer_orders = customers_df.merge(orders_df, on='customer_id', how='inner')
print(f"Result shape: {customer_orders.shape}")
print("Sample result:")
print(customer_orders[['customer_id', 'first_name', 'order_id', 'total_amount', 'order_status']].head(10))
#CREATING DATASET 2
print("Creating University Dataset...")


n_students = 500000
students_data = {
    'student_id': [f'STU_{i:05d}' for i in range(1, n_students + 1)],
    'first_name': [f'Student_{i}' for i in range(1, n_students + 1)],
    'last_name': [f'Surname_{i}' for i in range(1, n_students + 1)],
    'email': [f'student_{i}@university.edu' for i in range(1, n_students + 1)],
    'enrollment_year': np.random.choice([2020, 2021, 2022, 2023, 2024], n_students, p=[0.1, 0.15, 0.2, 0.25, 0.3]),
    'major': np.random.choice(['Computer Science', 'Business', 'Engineering', 'Mathematics', 'Biology', 'Psychology'], n_students),
    'gpa': np.random.normal(3.2, 0.6, n_students).clip(0, 4.0).round(2),
    'student_status': np.random.choice(['Active', 'Graduated', 'On Leave'], n_students, p=[0.75, 0.2, 0.05]),
    'scholarship_amount': np.random.choice([0, 1000, 2500, 5000, 10000], n_students, p=[0.5, 0.2, 0.15, 0.1, 0.05])
}

students_df = pd.DataFrame(students_data)


n_enrollments = 250000

probs_students = np.random.exponential(scale=0.3, size=n_students)
probs_students = probs_students / probs_students.sum()
student_enrollment_distribution = np.random.choice(
    students_df['student_id'].values,
    size=n_enrollments,
    replace=True,
    p=probs_students
)


course_codes = [f'{dept}_{num:03d}' for dept in ['CS', 'MATH', 'BUS', 'ENG', 'BIO', 'PSY']
                for num in range(100, 500, 10)]

enrollments_data = {
    'enrollment_id': range(1, n_enrollments + 1),
    'student_id': student_enrollment_distribution,
    'course_code': np.random.choice(course_codes, n_enrollments),
    'semester': np.random.choice(['Fall 2023', 'Spring 2024', 'Fall 2024'], n_enrollments, p=[0.3, 0.35, 0.35]),
    'grade': np.random.choice(['A', 'B', 'C', 'D', 'F', 'W'], n_enrollments, p=[0.25, 0.35, 0.25, 0.1, 0.03, 0.02]),
    'credits': np.random.choice([1, 3, 4, 6], n_enrollments, p=[0.1, 0.6, 0.25, 0.05]),
    'enrollment_date': pd.date_range('2023-08-15', periods=n_enrollments, freq='2H'),
    'final_score': np.random.normal(82, 12, n_enrollments).clip(0, 100).round(1)
}

enrollments_df = pd.DataFrame(enrollments_data)

print(f"Students DataFrame: {students_df.shape}")
print(f"Enrollments DataFrame: {enrollments_df.shape}")
print(f"Join Key: student_id")
print(f"Join Cardinality: 1-to-Many (One student can have multiple course enrollments)")
print()
#PERFORMING JOINS ON DATASET 2
print("\n3. Outer  Join Example (Students with Enrollments):")
student_enrollments = students_df.merge(enrollments_df, on='student_id', how='outer')
print(f"Result shape: {student_enrollments.shape}")
print("Sample result:")
print(student_enrollments[['student_id', 'first_name', 'major', 'course_code', 'grade', 'credits']].head(10))
filename = "output.csv"
df = cell_analyzer.get_dataframe()

# Save or append metrics csv
if df is not None:
    if os.path.exists(filename):
        df.to_csv(filename, mode='a', header=False, index=False)
        print(f"Appended data to existing file: {filename}")
    else:
        df.to_csv(filename, mode='w', header=True, index=False)
        print(f"Created new file and saved data: {filename}")

# Display dataframe
if df is not None:
    from IPython.display import display
    display(df)

# Show summary
cell_analyzer.show_summary()

if df is not None:
    print("3 slowest cells:")
    print(df.nlargest(3, 'execution_time_seconds')[['cell_number', 'execution_time_seconds', 'memory_delta_mb']])
    print(f"Total time: {df['execution_time_seconds'].sum():.1f}s")

# Generate and print Perplexity LLM summary
if df is not None:
    llm_summary_text = llm_generate_summary(df)
    print("\nPerplexity AI-Generated Summary:")
    print(llm_summary_text)
    cell_analyzer.send_all_alerts(llm_summary=llm_summary_text)

import matplotlib.pyplot as plt

def create_charts(df):
    if df is None or df.empty:
        print("No data available")
        return

    fig, axes = plt.subplots(1, 2, figsize=(24, 14))

    axes[0].bar(df['cell_number'], df['execution_time_seconds'])
    axes[0].set_title('Time Intensive Cells')
    axes[0].set_xlabel('Cell Number')
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_xticks(df['cell_number'])
    axes[0].set_xticklabels([f"C{int(x)}" for x in df['cell_number']], rotation=45, ha='right')

    colors = ['green' if x >= 0 else 'red' for x in df['memory_delta_mb']]
    axes[1].bar(df['cell_number'], df['memory_delta_mb'], color=colors)
    axes[1].set_title('Memory Intensive Cells')
    axes[1].set_xlabel('Cell Number')
    axes[1].set_ylabel('Memory (MB)')
    axes[1].set_xticks(df['cell_number'])
    axes[1].set_xticklabels([f"C{int(x)}" for x in df['cell_number']], rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

if df is not None:
    create_charts(df)
